In [ ]:
"""
Concepts:

1. Gradient Boosting
2. Learning Rate
3. Gamma
4. Min Child Weight
5. Subsample
6. Colsample By Tree
7. Regularization
8. Early Stopping
9. SHAP Explainability
10. Calibration Analysis
"""

# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split,cross_val_score,RandomizedSearchCV,learning_curve)
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report,roc_curve,precision_recall_curve)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

In [ ]:
# =====================================================
# STEP 1 : DATA UNDERSTANDING
# =====================================================

data = load_breast_cancer()
df = pd.DataFrame(data.data,columns=data.feature_names)
df["target"] = data.target
print(df.head())
print(df.info())
print(df.describe())
print(df["target"].value_counts())

In [ ]:
# =====================================================
# STEP 2 : EDA
# =====================================================

print(df.isnull().sum())
print(df.duplicated().sum())

In [ ]:
# =====================================================
# STEP 3 : TRAIN / VALIDATION / TEST
# =====================================================

X = df.drop("target", axis=1)
y = df["target"]

X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.15,random_state=42,stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42,stratify=y_temp)

# =====================================================
# STEP 4 : PREPROCESSING
# =====================================================

# XGBoost does NOT require scaling

In [ ]:
# =====================================================
# STEP 5 : BASELINE MODEL
# =====================================================

xgb = XGBClassifier(random_state=42,eval_metric="auc")
xgb.fit(X_train,y_train)

In [ ]:
# =====================================================
# STEP 6 : VALIDATION METRICS
# =====================================================

val_pred = xgb.predict(X_val)
val_prob = xgb.predict_proba(X_val)[:,1]

print("\nBASELINE XGBOOST")

accuracy_score(y_val,val_pred)
precision_score(y_val,val_pred)
recall_score(y_val,val_pred)
f1_score(y_val,val_pred)
roc_auc_score(y_val,val_prob)

In [ ]:
# =====================================================
# STEP 7 : CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(xgb,X_train,y_train,cv=5,scoring="roc_auc",n_jobs=-1)
print("\nCV Mean:",cv_scores.mean())
print("CV Std:",cv_scores.std())

In [ ]:
# =====================================================
# STEP 8 : HYPERPARAMETER TUNING
# =====================================================

param_dist = {
    "n_estimators":[200,300,500],
    "learning_rate":[0.01,0.05,0.1],
    "max_depth":[3,4,5,7],
    "min_child_weight":[1,3,5],
    "gamma":[0,0.1,0.3],
    "subsample":[0.7,0.8,1.0],
    "colsample_bytree":[0.7,0.8,1.0],
    "reg_alpha":[0,0.1,1],
    "reg_lambda":[1,5,10]}

random_search = RandomizedSearchCV(XGBClassifier(eval_metric='auc',random_state=42),
                                   param_dist,n_iter=30,cv=5,scoring='roc_auc',random_state=42,n_jobs=-1)

random_search.fit(X_train,y_train)
print(random_search.best_params_)
best_model = random_search.best_estimator_

In [ ]:
# =====================================================
# STEP 9 : VALIDATION RECHECK
# =====================================================

val_prob = best_model.predict_proba(X_val)[:,1]
val_auc = roc_auc_score(y_val,val_prob)
print("\nValidation AUC:",val_auc)

In [ ]:
# =====================================================
# STEP 10 : TRAIN VS VALIDATION
# =====================================================

train_prob = best_model.predict_proba(X_train)[:,1]
train_auc = roc_auc_score(y_train,train_prob)
print("Train AUC:",train_auc)
print("Validation AUC:",val_auc)

In [ ]:
# =====================================================
# STEP 11 : XGBOOST ANALYSIS
# =====================================================

# -----------------------------------------
# FEATURE IMPORTANCE
# -----------------------------------------

importance_df = pd.DataFrame({"Feature":X.columns,"Importance":best_model.feature_importances_}).sort_values(by="Importance",ascending=False)
print(importance_df.head(15))

# -----------------------------------------
# PERMUTATION IMPORTANCE
# -----------------------------------------

perm = permutation_importance(best_model,X_val,y_val,random_state=42)
print(perm.importances_mean)

# -----------------------------------------
# CONFUSION MATRIX
# -----------------------------------------

val_pred = best_model.predict(X_val)
cm = confusion_matrix(y_val,val_pred)
sns.heatmap(cm,annot=True,fmt="d")
plt.show()

# -----------------------------------------
# ROC CURVE
# -----------------------------------------

fpr,tpr,_ = roc_curve(y_val,val_prob)
plt.plot(fpr,tpr)
plt.plot([0,1],[0,1])
plt.show()

# -----------------------------------------
# PRECISION RECALL CURVE
# -----------------------------------------

precision, recall, _ = precision_recall_curve(y_val,val_prob)
plt.plot(recall,precision)
plt.show()

# -----------------------------------------
# CALIBRATION CURVE
# -----------------------------------------

prob_true, prob_pred = calibration_curve(y_val,val_prob,n_bins=10)
plt.plot(prob_pred,prob_true,marker='o')
plt.plot([0,1],[0,1])
plt.show()

# -----------------------------------------
# THRESHOLD OPTIMIZATION
# -----------------------------------------

thresholds = np.arange(0.1,0.91,0.05)
f1_scores = []

for threshold in thresholds:
    pred = (val_prob >= threshold).astype(int)
    score = f1_score(y_val,pred)
    f1_scores.append(score)

plt.plot(thresholds,f1_scores,marker='o')
plt.title("Threshold Optimization")
plt.show()

# -----------------------------------------
# LEARNING CURVE
# -----------------------------------------

train_sizes, train_scores, val_scores = learning_curve(best_model,X_train,y_train,cv=5,scoring='roc_auc',n_jobs=-1)

plt.plot(train_sizes,np.mean(train_scores,axis=1),label="Train")
plt.plot(train_sizes,np.mean(val_scores,axis=1),label="Validation")
plt.legend()
plt.show()


In [ ]:
# =====================================================
# STEP 12 : FINAL MODEL
# =====================================================

final_model = best_model

In [ ]:
# =====================================================
# STEP 13 : TEST EVALUATION
# =====================================================

test_pred = final_model.predict(X_test)
test_prob = final_model.predict_proba(X_test)[:,1]
print(classification_report(y_test,test_pred))
print("Test ROC-AUC:",roc_auc_score(y_test,test_prob))

In [ ]:
# =====================================================
# STEP 14 : DEPLOYMENT READINESS
# =====================================================

print("CV AUC:",cv_scores.mean())
print("Validation AUC:",val_auc)
print("Test AUC:",roc_auc_score(y_test,test_prob))

# =====================================================
# STEP 15 : INTERVIEW QUESTIONS
# =====================================================

"""
1. What is boosting?
2. Difference between Bagging and Boosting?
3. Why XGBoost performs better than RF?
4. What is learning_rate?
5. Why lower learning_rate needs more trees?
6. What is gamma?
7. What is min_child_weight?
8. What is subsample?
9. What is colsample_bytree?
10. What is regularization in XGBoost?
11. Difference between reg_alpha and reg_lambda?
12. What is early stopping?
13. Why does XGBoost resist overfitting?
14. What is feature importance?
15. What is gain?
16. What is cover?
17. What is weight importance?
18. Why XGBoost handles missing values well?
19. Why XGBoost is popular in Kaggle?
20. Explain XGBoost end-to-end.
"""